** Notes:** 
* This ipynb gets twitter users' geolocation data using the official Twitter API--- tweepy.
* **Input files** come from ../1-downloadTweets/tweets-data in the form of *month*.csv
* output files are saved to ./out-userGeolocations/
* main **output file** is twitter-users-jan2018-6nov2018.csv

In [ ]:
%load_ext jupyternotify

import pandas as pd
import numpy as np
import os.path
import tweepy

from tweepy import OAuthHandler
from tqdm import tqdm

In [ ]:
def get_user_data(dataframe, start, end):
    for ix in tqdm(range(start, end)):
        if dataframe.at[ix, 'status'] == 0:
            username = dataframe.at[ix, 'username']

            try:
                userOBJ = api.get_user(username)
            except:
                dataframe.at[ix, 'user_not_found'] = 1
                dataframe.at[ix, 'location'] = np.nan
                dataframe.at[ix, 'status'] = 1
                continue

            if userOBJ.location != '':
                dataframe.at[ix, 'location'] = userOBJ.location
            else:
                dataframe.at[ix, 'location'] = np.nan
                dataframe.at[ix, 'loc_not_found'] = 1

            dataframe.at[ix, 'status'] = 1
        else: pass
    
    return dataframe

In [ ]:
CONSUMER_KEY = 'ikTc2nEVjASF3MB9jcxAMCACy'
CONSUMER_SECRET = 'kE0dp8PRco2vvoBK210qs9wAAYYud4lKfYwBjE8iSXUng7In8B'
ACCESS_KEY = '1071664120544587777-7WWDqzCNurbxKVso66kcANAAZRt9Yw'
ACCESS_SECRET = 'hgGgKTXsiaedzhRUT0nmTMrSQ8Lhqu5dk8HSAL0mp0BFz'

In [ ]:
auth = tweepy.OAuthHandler(CONSUMER_KEY, CONSUMER_SECRET)
auth.set_access_token(ACCESS_KEY, ACCESS_SECRET)

api = tweepy.API(auth, wait_on_rate_limit=True, wait_on_rate_limit_notify=True)

### Get tweets user data for Oct2018 to Nov2018 (1st to 5th Nov)

In [ ]:
folderpath = os.path.join('..', '1-downloadTweets', 'tweets-data')

filepath = os.path.join(folderpath, 'oct2018.csv')
oct_df = pd.read_csv(filepath, index_col=0)

filepath = os.path.join(folderpath, 'nov2018.csv')

nov_df = pd.read_csv(filepath, index_col=0)

df = pd.concat([oct_df, nov_df], axis=0, ignore_index=True)
df = df[['username', 'author_id']]
df.drop_duplicates(subset=df.columns, keep='first', inplace=True)

df.reset_index(inplace=True, drop=True)
df['location'] = ''
df['status'] = 0

len(df)

In [ ]:
for ix in tqdm(range(len(df))):
    if df.at[ix, 'status'] == 0:
        username = df.at[ix, 'username']
        
        try:
            userOBJ = api.get_user(username)
        except:
            df.at[ix, 'user_not_found'] = 1
            df.at[ix, 'location'] = np.nan
            df.at[ix, 'status'] = 1
            continue
        
        if userOBJ.location != '':
            df.at[ix, 'location'] = userOBJ.location
        else:
            df.at[ix, 'location'] = np.nan
            df.at[ix, 'loc_not_found'] = 1
            
        df.at[ix, 'status'] = 1
    else: pass

In [ ]:
saveFolderPath = os.join('', 'out-userGeolocations')
df.to_csv(os.path.join(saveFolderPath, 'twitter-users.csv'))

### Get tweets for Jan2018-Sep2018

In [ ]:
if False:
    folderpath = os.path.join('..', 'tweets-data')

    dfJan2018 = pd.read_csv(os.path.join(folderpath, 'jan2018.csv'), usecols=['username', 'author_id'])
    dfFeb2018 = pd.read_csv(os.path.join(folderpath, 'feb2018.csv'), usecols=['username', 'author_id'])
    dfMar2018 = pd.read_csv(os.path.join(folderpath, 'mar2018.csv'), usecols=['username', 'author_id'])
    dfApr2018 = pd.read_csv(os.path.join(folderpath, 'apr2018.csv'), usecols=['username', 'author_id'])
    dfMay2018 = pd.read_csv(os.path.join(folderpath, 'may2018.csv'), usecols=['username', 'author_id'])
    dfJun2018 = pd.read_csv(os.path.join(folderpath, 'jun2018.csv'), usecols=['username', 'author_id'])
    dfJul2018 = pd.read_csv(os.path.join(folderpath, 'jul2018.csv'), usecols=['username', 'author_id'])
    dfAug2018 = pd.read_csv(os.path.join(folderpath, 'aug2018.csv'), usecols=['username', 'author_id'])
    dfSep2018 = pd.read_csv(os.path.join(folderpath, 'sep2018.csv'), usecols=['username', 'author_id'])

    dfJantoSep2018Users = dfJan2018.append([dfFeb2018, dfMar2018, dfApr2018, dfMay2018, dfJun2018, 
                                            dfJul2018, dfAug2018, dfSep2018], ignore_index=True)

    dfJantoSep2018Users.drop_duplicates(subset=dfJantoSep2018Users.columns, keep='first', inplace=True)

    dfJantoSep2018Users['location'] = ''
    dfJantoSep2018Users['status'] = 0
    dfJantoSep2018Users.reset_index(inplace=True, drop=True)

    dfJantoSep2018Users.to_csv(os.path.join(saveFolderPath, 'twitter-users-jan2018-sep2018.csv'), index=False)

In [ ]:
get_user_data(dfJantoSep2018Users, 250000, 300000)

In [ ]:
dfJantoSep2018Users.iloc[250000:300000]

In [ ]:
for ix in range(150000, len(dfJantoSep2018Users)):
    dfJantoSep2018Users.set_value(ix, 'status', 0)
    dfJantoSep2018Users.set_value(ix, 'loc_not_found', np.nan)
    dfJantoSep2018Users.set_value(ix, 'user_not_found', np.nan)

In [ ]:
dfJantoSep2018Users.to_csv(os.path.join(saveFolderPath, 'twitter-users-jan2018-sep2018.csv'), index=False)

In [ ]:
len(dfJantoSep2018Users)

In [ ]:
dfJantoSep2018Users.tail()

### Check

In [ ]:
folderpath = os.path.join('..', 'tweets-data')
df = pd.read_csv(os.path.join(folderpath, 'twitter-users-jan2018-6nov2018.csv'))

In [ ]:
_df = df[df['location'].isnull()]
_df['status'] = 0
_df['loc_not_found'] = 0
_df['user_not_found'] = 0
_df.reset_index(drop=True, inplace=True)

In [ ]:
get_user_data(_df, 0, len(_df))

In [ ]:
dfWithOldData = df[(df['user_not_found'].isnull()) & (df['loc_not_found'].isnull())]
dfNewToSave = dfWithOldData.append(_df, ignore_index=True)
dfNewToSave.to_csv(os.path.join(folderpath, 'twitter-users-jan2018-6nov2018.csv'), index=False)

### Deal with duplicates
* probably from the Jan0218 to Sep2018 and the Oct2018 to Nov2018

In [ ]:
file = os.path.join('out-userGeolocations', 'twitter-users-jan2018-6nov2018.csv')

In [ ]:
df = pd.read_csv(file)

len(df)

In [ ]:
df.drop_duplicates('username', keep='first', inplace=True)

len(df)

In [ ]:
df.to_csv(file, index=False)

### Get post-election geolocation

In [ ]:
folderPath = os.path.join('..', '1-downloadTweets', 'tweets-data', 'post-election')
saveFolderPath = os.path.join('', 'out-userGeolocations')

In [ ]:
if False:
    dfNov2018 = pd.read_csv(os.path.join(folderPath, 'nov2018.csv'), usecols=['username', 'author_id'])
    dfJan2019 = pd.read_csv(os.path.join(folderPath, 'jan2019.csv'), usecols=['username', 'author_id'])
    dfFeb2019 = pd.read_csv(os.path.join(folderPath, 'feb2019.csv'), usecols=['username', 'author_id'])
    dfMar2019 = pd.read_csv(os.path.join(folderPath, 'mar2019.csv'), usecols=['username', 'author_id'])
    dfApr2019 = pd.read_csv(os.path.join(folderPath, 'apr2019.csv'), usecols=['username', 'author_id'])
    dfMay2019 = pd.read_csv(os.path.join(folderPath, 'may2019.csv'), usecols=['username', 'author_id'])


    df2018 = dfNov2018.append([dfJan2019, dfFeb2019, dfMar2019, dfApr2019, dfMay2019], ignore_index=True)

    df2018.drop_duplicates(subset=df2018.columns, keep='first', inplace=True)

    df2018['status'] = 0
    df2018.reset_index(inplace=True, drop=True)

    df2018.to_csv(os.path.join(saveFolderPath, 'post-election-twitter-users.csv'), index=False)

In [ ]:
df2018 = pd.read_csv(os.path.join(saveFolderPath, 'post-election-twitter-users.csv'))
len(df2018)

In [ ]:
get_user_data(df2018, 220000, len(df2018))

In [ ]:
df2018.to_csv(os.path.join(saveFolderPath, 'post-election-twitter-users.csv'), index=False)